# CONFIG

In [1]:
import os

print(os.getcwd())
if not os.getcwd().endswith("app"):
    os.chdir("../app")
    print(os.getcwd())

import pandas as pd
pd.set_option('display.max_rows', 500)
pd.set_option('display.max_columns', 500)


%load_ext autoreload
%autoreload 2
%matplotlib inline

/home/turbotowerlnx/Documents/Master/BIOM/notebooks
/home/turbotowerlnx/Documents/Master/BIOM/app


In [2]:
import numpy as np
from tqdm import tqdm

from maikol_utils.print_utils import print_separator, print_warn, print_color

from src.config import Configuration    


CONFIG = Configuration()

# seed all
np.random.seed(CONFIG.seed)

# GET ALL POSSIBLE HAAR

- We no dot create, for a single HAAR, both versions white-black / black-white, only one
- Since one is just the other $\times -1$, the weak classifiers will handle that by multiplying by $1$ or $-1$ as needed 

In [3]:
from typing import List
from src.model import Feature, Rectangle


def generate_all_features(win_w: int = 24, win_h: int = 24) -> List[Feature]:
    features = []
    
    for w in range(1, win_w + 1):
        for h in range(1, win_h + 1):
            for x in range(win_w - w + 1):
                for y in range(win_h - h + 1):
                    
                    # 2-rectangle horizontal (Left/Right)
                    if w % 2 == 0:
                        w_half = w // 2
                        features.append(Feature(
                            feature_id = len(features),
                            rectangles = [
                            Rectangle(x, y, w_half, h, -1.0),
                            Rectangle(x + w_half, y, w_half, h, 1.0)
                        ]))
                        
                    # 2-Rectangle vertical (Top/Bottom)
                    if h % 2 == 0:
                        h_half = h // 2
                        features.append(Feature(
                            feature_id = len(features),
                            rectangles = [
                            Rectangle(x, y, w, h_half, -1.0),
                            Rectangle(x, y + h_half, w, h_half, 1.0)
                        ]))
                        
                    # # 3-Rectangle horizontal (Left/Center/Right)
                    # if w % 3 == 0:
                    #     w_third = w // 3
                    #     features.append(Feature(
                    #         feature_id = len(features),
                    #         rectangles = [
                    #         Rectangle(x, y, w_third, h, -1.0),
                    #         Rectangle(x + w_third, y, w_third, h, 2.0), # Center weight compensates for 2 outside Rectangles
                    #         Rectangle(x + 2 * w_third, y, w_third, h, -1.0)
                    #     ]))
                        
                    # # 3-Rectangle vertical (Top/Center/Bottom)
                    # if h % 3 == 0:
                    #     h_third = h // 3
                    #     features.append(Feature(
                    #         feature_id = len(features),
                    #         rectangles = [
                    #         Rectangle(x, y, w, h_third, -1.0),
                    #         Rectangle(x, y + h_third, w, h_third, 2.0),
                    #         Rectangle(x, y + 2 * h_third, w, h_third, -1.0)
                    #     ]))
                        
                    # # 4-Rectangle (Checkerboard)
                    # if w % 2 == 0 and h % 2 == 0:
                    #     w_half, h_half = w // 2, h // 2
                    #     features.append(Feature(
                    #         feature_id = len(features),
                    #         rectangles = [
                    #         Rectangle(x, y, w_half, h_half, 1.0),
                    #         Rectangle(x + w_half, y, w_half, h_half, -1.0),
                    #         Rectangle(x, y + h_half, w_half, h_half, -1.0),
                    #         Rectangle(x + w_half, y + h_half, w_half, h_half, 1.0)
                    #     ]))
                        
    return features

In [4]:
all_features = generate_all_features(
    win_w = CONFIG.crop_size, 
    win_h = CONFIG.crop_size
)

# LOAD DATA

In [5]:
import fiftyone as fo
import fiftyone.zoo as foz
from fiftyone import ViewField as F

from maikol_utils.file_utils import load_json, list_dir_files

# =============== FACES DATASET =============== 
all_faces, n = list_dir_files(CONFIG.faces_path, recursive=True)
print(f"Found {n} files in {CONFIG.faces_path}")


# =============== NO-FACES DATASET =============== 
to_keep_labels = load_json(CONFIG.dataset_classes_path)

# Download dataset without faces
bg_dataset = foz.load_zoo_dataset(
    "open-images-v7",
    split="train",
    label_types=["detections"],
    classes=to_keep_labels,
    max_samples=5000,
    dataset_dir=CONFIG.no_faces_path
)
bg_dataset = bg_dataset.filter_labels("ground_truth", F("label").is_in(to_keep_labels))


/home/turbotowerlnx/Documents/Master/BIOM/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Found 136965 files in ../data/ViolaJones/face_images
Loading output from ../data/dataset_classes.json...
 100% |██████|    4.8Gb/4.8Gb [13.2s elapsed, 0s remaining, 370.3Mb/s]      
 100% |█████████████████| 5000/5000 [1.3m elapsed, 0s remaining, 70.1 files/s]      
Dataset info written to '/home/turbotowerlnx/fiftyone/open-images-v7/info.json'
Loading existing dataset 'open-images-v7-train-5000'. To reload from disk, either delete the existing dataset or provide a custom `dataset_name` to use


Dataset:     open-images-v7-train-5000
Media type:  image
Num samples: 10
Sample fields:
    id:               fiftyone.core.fields.ObjectIdField
    filepath:         fiftyone.core.fields.StringField
    tags:             fiftyone.core.fields.ListField(fiftyone.core.fields.StringField)
    metadata:         fiftyone.core.fields.EmbeddedDocumentField(fiftyone.core.metadata.ImageMetadata)
    created_at:       fiftyone.core.fields.DateTimeField
    last_modified_at: fiftyone.core.fields.DateTimeField
    ground_truth:     fiftyone.core.fields.EmbeddedDocumentField(fiftyone.core.labels.Detections)
View stages:
    1. FilterLabels(field='ground_truth', filter={'$in': ['$$this.label', [...]]}, only_matches=True, trajectories=False)
    2. Limit(limit=10)

### PRECOMPUTE ALL FEATURES FOR FACES DATASET

In [ ]:
all_faces = np.random.choice(all_faces, size=25000, replace=False)
print(f"Using {len(all_faces)} files in {CONFIG.faces_path}")

Using 25000 files in ../data/ViolaJones/face_images
Using 50000 files in ../data/ViolaJones/crops


In [ ]:
import os
import cv2
import multiprocessing as mp
from src.data import get_integral_image, get_integral_squared_image, get_std_from_integral_images
from src.model import compute_feature

def extract_features(img_path: str = None, img: np.ndarray = None) -> np.ndarray:
    if img is None:
        img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)

    integral_img = get_integral_image(img)
    integral_img_2 = get_integral_squared_image(img)

    # Calculate std
    H, W = img.shape
    r1 = np.array([[0]])
    r2 = np.array([[H - 1]])
    c1 = np.array([[0]])
    c2 = np.array([[W - 1]])

    _, std_dev = get_std_from_integral_images(integral_img, integral_img_2, r1, r2, c1, c2)
    std_dev = std_dev[0, 0]

    if std_dev <= 0:
        std_dev = 1.0

    # FIX: Divide by std_dev (not multiply) for proper contrast normalization per Viola-Jones
    features = [compute_feature(integral_img, feature) / std_dev for feature in all_features]
    return np.asarray(features, dtype=np.float32)

def compute_features_dataset(images_paths):
    n_workers = int(max(1, (os.cpu_count() or 1) - 1) * 4 / 5)
    chunksize = 8

    with mp.get_context("fork").Pool(processes=n_workers) as pool:
        results_faces = list(
            tqdm(
                pool.imap(extract_features, images_paths, chunksize=chunksize),
                total=len(images_paths),
                desc=f"Extracting face features ({n_workers} workers)",
            )
        )

    features = np.stack(results_faces, axis=0).astype(np.float32, copy=False)
    return features


X_train_faces = compute_features_dataset(all_faces)
np.save(CONFIG.faces_np_path, X_train_faces)
print(f"Computed face features for {X_train_faces.shape[0]} images.")
print(f"X_train_faces dtype={X_train_faces.dtype}, shape={X_train_faces.shape}")

Extracting non-face features (18 workers): 100%|██████████| 50000/50000 [12:18<00:00, 67.74it/s] 


Computed face features for 25000 images.
X_train_faces dtype=float32, shape=(25000, 86400)


### Load values again

In [ ]:
X_train_faces = np.load(CONFIG.faces_np_path)

# VIOLA JONES STAGES

### Train AdaBoost
- Train with stumps, using at most x features per stage
- Then check at which point of the trained, we mee the True positive and False positive rate
- Remove unused stages 
- Return remaining tree and threshold for that stage

In [ ]:
from sklearn.ensemble import AdaBoostClassifier
from sklearn.tree import DecisionTreeClassifier
import numpy as np

def train_stage_early_stopping(X_train, y_train, max_features=200, target_tpr=0.995, target_fpr=0.50):
    """
    train_stage_for_tpr trains an AdaBoost classifier and determines a custom threshold to achieve the target true positive rate (TPR) on the training data.

    X_train shape: (num_images, num_features) - precalculated feature values
    y_train shape: (num_images,) - 1 for face, 0 for background

    returns:
    - clf: the trained AdaBoost classifier
    - custom_threshold: the decision function threshold to achieve the target TPR on the training data
    """
    print(' - Fitting AdaBoost with early stopping...')
    clf = AdaBoostClassifier(
        estimator=DecisionTreeClassifier(max_depth=1),
        n_estimators=max_features, # set to a cap for bc no need to check all
    )
    clf.fit(X_train, y_train)

    X_faces = X_train[y_train == 1]
    X_bg = X_train[y_train == 0]

    print(' - Refining threshold and selecting features...')
    # .staged_decision_function evaluates the ensemble using 1 feature, then 2 features, etc.
    for i, (face_scores, bg_scores) in enumerate(zip(
        clf.staged_decision_function(X_faces),
        clf.staged_decision_function(X_bg)
    )):
        
        # Force the threshold to meet the 99.5% TPR target
        face_scores_sorted = np.sort(face_scores)
        drop_count = int(len(face_scores) * (1.0 - target_tpr))
        custom_threshold = face_scores_sorted[drop_count]

        # Check the False Positive Rate using this forced threshold
        false_positives = np.sum(bg_scores >= custom_threshold)
        fpr = false_positives / len(X_bg)

        print(f"   - Features: {i+1} | FPR: {fpr:.3f}")

        # 3. Stop early if we hit the FPR target!
        if fpr <= target_fpr:
            print(f" - Stage criteria met! Stopping at {i+1} features.")
            
            # Truncate the sklearn classifier to drop the unused extra features
            clf.estimators_ = clf.estimators_[:i+1]
            clf.estimator_weights_ = clf.estimator_weights_[:i+1]
            clf.estimator_errors_ = clf.estimator_errors_[:i+1]
            clf.classes_ = np.array([0, 1])
            
            return clf, custom_threshold

    print_warn("Max features reached without hitting FPR target.")
    return clf, custom_threshold

# To predict later: 
# passes_stage = clf.decision_function(X_test) >= custom_threshold

### Generate hard mining samples
- Load random images without faces and get crops that PASS all the current stages


In [ ]:
from src.model import build_haar_cascade_from_stages, CascadeClassifier
from src.data import get_image_crops_from_list

def balance_non_face_samples(stages, num_samples, bg_dataset, =10000): 
    cascade = build_haar_cascade_from_stages(
        stages_output=stages,
        all_features=all_features,
        width=CONFIG.crop_size,
        height=CONFIG.crop_size,
        cascade_type="trained_adaboost_stages",
        feature_type="HAAR",
    )
    classifier = CascadeClassifier(CONFIG, cascade)
    
    print(f" - Generating {num_samples} hard negative samples...")
    all_hard_bg = []
    
    iterations = 0
    
    while len(all_hard_bg) < num_samples and iterations < max_iterations:
        iterations += 1
        bg_sample = np.random.choice(bg_dataset, replace=False)
        fps = classifier.predict(img_path=bg_sample.filepath)
        if fps:  # Only add if cascade found false positives
            all_hard_bg.append((fps, bg_sample.filepath))

    if len(all_hard_bg) < num_samples:
        print_warn(f"Only found {len(all_hard_bg)} hard negative samples (requested {num_samples}) after {iterations} iterations.")

    all_hard_bg = [
        extract_features(img=crop["img"]) 
        for fps, path in all_hard_bg
        for crop in get_image_crops_from_list(fps, img_path=path)
    ]

    return np.array(all_hard_bg, dtype=np.float32)

### Generate all cascade stages
2 stopping criteria 
- reaching FPR of $10 e -6$. This is computed not as in the paper (they estimated it from each stage $fpr_i$), but rather from the number of windows that had to be scanned to create the new 'X_train_bg'.
- Hand number (say 50)


In [ ]:
def generate_all_stages(X_train_faces, bg_dataset, max_stages=10, target_fpr=0.005):
    stages = []
    prev_n_faces = len(X_train_faces) 
    n_bg_pre = len(X_train_faces) 
    prev_fp = np.array([])
    fpr_macro = 1.0

    for stage_num in range(max_stages):
        print_separator(f"Training stage {stage_num + 1}/{max_stages}", sep_type="LONG")
        print_separator("Generating hard negative samples")

        X_train_bg = balance_non_face_samples(
            stages, 
            num_samples=prev_n_faces - len(prev_fp), # Only add enough new bg samples to maintain balance with faces
            bg_dataset=bg_dataset
        )

        X_train = np.vstack((X_train_faces, X_train_bg, prev_fp))
        y_train = np.hstack((np.ones(len(X_train_faces)), np.zeros(len(X_train_bg)), np.zeros(len(prev_fp))))
        del X_train_bg 

        print_separator("Training")
        clf, threshold = train_stage_early_stopping(X_train, y_train)
        stages.append((clf, threshold))
        
        print_separator("Filtering: Hard Negative mining")
        decision_scores = clf.decision_function(X_train)
        passes_stage = decision_scores >= threshold
        X_train = X_train[passes_stage]
        y_train = y_train[passes_stage]
        
        prev_fp = X_train[y_train == 0]

        n_faces = np.sum(y_train == 1)
        n_bg = len(prev_fp)
        # macro FPR: Fi = Fi-1 * fpr_micro with F0 = 1.0  
        fpr_micro = n_bg / n_bg_pre
        fpr_macro *= fpr_micro

        print(f" - Stage {stage_num + 1} used {len(clf.estimators_)} features.")
        print(f" - After stage {stage_num + 1}, {len(X_train)} samples remain for training.")
        print(f"   - {n_faces} faces")
        print(f"   - {n_bg} non-faces")
        print(f"   - {fpr_micro:.4f} micro false positive rate")
        print(f"   - {fpr_macro:.4f} macro false positive rate")
        
        n_bg_pre = n_bg
        prev_n_faces = n_faces
        # Stop conditions
        if n_bg == 0:
            print_color("No more negative samples left. Stopping training.", color="green")
            break
        
        if fpr_macro <= target_fpr:
            print_color(f"Reached target FPR of {target_fpr:.4f}. Stopping training.", color="green")
            break

    return stages, fpr_macro




stages, fpr_macro = generate_all_stages(X_train_faces, bg_dataset, max_stages=CONFIG.max_stages, target_fpr=CONFIG.objective_fpr)

________________________________________________________________________________________________________________________________
                                                     Training stage 1/50...                                                     

- Training...


# FINAL STAGES ClASSIFIER

### Save the trained cascade to XML

In [ ]:
from src.model import CascadeSerializer

haar_cascade = build_haar_cascade_from_stages(
    stages_output=stages,
    all_features=all_features,
    width=CONFIG.crop_size,
    height=CONFIG.crop_size,
    cascade_type="trained_adaboost_stages",
    feature_type="HAAR",
)

print_separator(f"FINAL CASCADE", sep_type="LONG")
print(f" - Stages: {len(haar_cascade.stages)}")
print(f" - Features used: {len(haar_cascade.features)}")
print(f" - Window size: {haar_cascade.width}x{haar_cascade.height}")
print(haar_cascade)


cascade_path = os.path.join(CONFIG.computed_haar_cascades, f"stages_vj-{fpr_macro:.4f}.xml")
CascadeSerializer.save(haar_cascade, cascade_path)

print(f" - File: {cascade_path}")



In [ ]:

loaded_cascade = CascadeSerializer.load(cascade_path)